# Sports Injury Analysis: Recovery Patterns Study

## Research Question
**What factors influence injury recovery time across different sports and injury types?**

### Hypotheses
1. **H1**: ACL injuries result in significantly longer recovery times than hamstring injuries
2. **H2**: Injuries during playoffs require longer recovery than regular season injuries
3. **H3**: Recurring injuries have longer recovery times than first-time injuries

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Set style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

## 1. Data Loading and Exploration

In [ ]:
# Load injury data
df = pd.read_csv('../data/player_injuries.csv')

print(f"Dataset Shape: {df.shape}")
print(f"\nColumns: {list(df.columns)}")
df.head(10)

In [ ]:
# Summary statistics
print("Injury Data Summary:")
print(f"\nSports represented: {df['sport'].unique().tolist()}")
print(f"Injury types: {df['injury_type'].unique().tolist()}")
print(f"\nDays missed statistics:")
print(df['days_missed'].describe())

## 2. Injury Type Analysis

In [ ]:
# Recovery time by injury type
injury_stats = df.groupby('injury_type').agg({
    'days_missed': ['count', 'mean', 'std', 'median'],
    'surgery_required': 'mean'
}).round(2)

injury_stats.columns = ['count', 'mean_days', 'std_days', 'median_days', 'surgery_rate']
injury_stats = injury_stats.sort_values('mean_days', ascending=False)

print("Recovery Time by Injury Type:")
injury_stats

In [ ]:
# Visualize injury types
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Bar chart of mean recovery time
colors = plt.cm.RdYlGn_r(np.linspace(0.2, 0.9, len(injury_stats)))
injury_stats['mean_days'].plot(kind='barh', ax=axes[0], color=colors, edgecolor='white')
axes[0].set_xlabel('Mean Days Missed')
axes[0].set_title('Average Recovery Time by Injury Type')

# Box plot
order = df.groupby('injury_type')['days_missed'].median().sort_values(ascending=False).index
sns.boxplot(data=df, y='injury_type', x='days_missed', order=order, ax=axes[1], palette='RdYlGn_r')
axes[1].set_xlabel('Days Missed')
axes[1].set_ylabel('Injury Type')
axes[1].set_title('Recovery Time Distribution')

plt.tight_layout()
plt.savefig('../visualizations/injury_types.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. H1: ACL vs Hamstring Recovery

In [ ]:
# Compare ACL and hamstring injuries
acl_data = df[df['injury_type'] == 'Knee ACL']['days_missed']
hamstring_data = df[df['injury_type'] == 'Hamstring']['days_missed']

print("H1: ACL vs Hamstring Recovery Time")
print("="*50)
print(f"\nACL Injuries: n={len(acl_data)}, mean={acl_data.mean():.1f} days, median={acl_data.median():.0f} days")
print(f"Hamstring Injuries: n={len(hamstring_data)}, mean={hamstring_data.mean():.1f} days, median={hamstring_data.median():.0f} days")

In [ ]:
# Independent t-test
t_stat, p_value = stats.ttest_ind(acl_data, hamstring_data, equal_var=False)

print(f"\nWelch's t-test:")
print(f"t-statistic: {t_stat:.4f}")
print(f"p-value: {p_value:.2e}")

# Effect size
pooled_std = np.sqrt(((len(acl_data)-1)*acl_data.std()**2 + (len(hamstring_data)-1)*hamstring_data.std()**2) / 
                       (len(acl_data) + len(hamstring_data) - 2))
cohens_d = (acl_data.mean() - hamstring_data.mean()) / pooled_std

print(f"Cohen's d: {cohens_d:.3f}")

if p_value < 0.05 and acl_data.mean() > hamstring_data.mean():
    print("\n✓ H1 SUPPORTED: ACL injuries require significantly longer recovery")
else:
    print("\n✗ H1 NOT SUPPORTED")

In [ ]:
# Visualize comparison
fig, ax = plt.subplots(figsize=(10, 6))

data_to_plot = [acl_data, hamstring_data]
bp = ax.boxplot(data_to_plot, labels=['ACL Injury', 'Hamstring Injury'], patch_artist=True)

colors = ['#e74c3c', '#3498db']
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

ax.set_ylabel('Days Missed')
ax.set_title(f'ACL vs Hamstring Recovery Time\n(t={t_stat:.2f}, p={p_value:.2e}, d={cohens_d:.2f})')

# Add means
ax.scatter([1, 2], [acl_data.mean(), hamstring_data.mean()], 
           marker='D', color='gold', s=100, zorder=5, label='Mean')
ax.legend()

plt.tight_layout()
plt.savefig('../visualizations/acl_vs_hamstring.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. H2: Season Phase Impact

In [ ]:
# Recovery by season phase
phase_stats = df.groupby('season_phase')['days_missed'].agg(['count', 'mean', 'std', 'median'])
print("Recovery Time by Season Phase:")
print(phase_stats.round(1))

In [ ]:
# One-way ANOVA
phases = df['season_phase'].unique()
phase_groups = [df[df['season_phase'] == phase]['days_missed'] for phase in phases]

f_stat, anova_p = stats.f_oneway(*phase_groups)

print(f"\nOne-Way ANOVA:")
print(f"F-statistic: {f_stat:.4f}")
print(f"p-value: {anova_p:.4f}")

# Compare playoffs vs regular season specifically
playoffs = df[df['season_phase'] == 'Playoffs']['days_missed']
regular = df[df['season_phase'] == 'Regular Season']['days_missed']

t_stat2, p_val2 = stats.ttest_ind(playoffs, regular, equal_var=False)

print(f"\nPlayoffs vs Regular Season t-test:")
print(f"t-statistic: {t_stat2:.4f}")
print(f"p-value: {p_val2:.4f}")

if p_val2 < 0.05 and playoffs.mean() > regular.mean():
    print("\n✓ H2 SUPPORTED: Playoff injuries require longer recovery")
elif p_val2 >= 0.05:
    print("\n✗ H2 NOT SUPPORTED: No significant difference found")
else:
    print("\n✗ H2 NOT SUPPORTED: Playoffs actually have shorter recovery")

In [ ]:
# Visualize season phase
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Distribution comparison
for i, phase in enumerate(['Preseason', 'Regular Season', 'Playoffs']):
    data = df[df['season_phase'] == phase]['days_missed']
    sns.kdeplot(data, label=phase, fill=True, alpha=0.5, ax=axes[0])

axes[0].set_xlabel('Days Missed')
axes[0].set_title('Recovery Time Distribution by Season Phase')
axes[0].legend()

# Box plot
sns.boxplot(data=df, x='season_phase', y='days_missed', 
            order=['Preseason', 'Regular Season', 'Playoffs'],
            palette='Set2', ax=axes[1])
axes[1].set_xlabel('Season Phase')
axes[1].set_ylabel('Days Missed')
axes[1].set_title('Recovery Time by Season Phase')

plt.tight_layout()
plt.savefig('../visualizations/season_phase_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. H3: Recurring vs First-Time Injuries

In [ ]:
# Compare recurring vs first-time
recurring = df[df['recurring'] == True]['days_missed']
first_time = df[df['recurring'] == False]['days_missed']

print("H3: Recurring vs First-Time Injuries")
print("="*50)
print(f"\nRecurring: n={len(recurring)}, mean={recurring.mean():.1f} days")
print(f"First-time: n={len(first_time)}, mean={first_time.mean():.1f} days")
print(f"Difference: {recurring.mean() - first_time.mean():.1f} days")

In [ ]:
# t-test
t_stat3, p_val3 = stats.ttest_ind(recurring, first_time, equal_var=False)

print(f"\nWelch's t-test:")
print(f"t-statistic: {t_stat3:.4f}")
print(f"p-value: {p_val3:.4f}")

if p_val3 < 0.05 and recurring.mean() > first_time.mean():
    print("\n✓ H3 SUPPORTED: Recurring injuries require longer recovery")
else:
    print("\n✗ H3 NOT SUPPORTED")

In [ ]:
# Visualize recurring vs first-time
fig, ax = plt.subplots(figsize=(10, 6))

# Stacked histogram by injury type
recurring_by_type = df[df['recurring'] == True].groupby('injury_type')['days_missed'].mean()
first_by_type = df[df['recurring'] == False].groupby('injury_type')['days_missed'].mean()

x = np.arange(len(recurring_by_type))
width = 0.35

ax.bar(x - width/2, first_by_type.values, width, label='First-time', color='#3498db', alpha=0.8)
ax.bar(x + width/2, recurring_by_type.values, width, label='Recurring', color='#e74c3c', alpha=0.8)

ax.set_ylabel('Mean Days Missed')
ax.set_title('Recovery Time: First-time vs Recurring by Injury Type')
ax.set_xticks(x)
ax.set_xticklabels(recurring_by_type.index, rotation=45, ha='right')
ax.legend()

plt.tight_layout()
plt.savefig('../visualizations/recurring_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Sport Comparison

In [ ]:
# Recovery by sport
sport_stats = df.groupby('sport').agg({
    'days_missed': ['count', 'mean', 'median'],
    'surgery_required': 'mean',
    'recovery_success': 'mean'
}).round(2)

sport_stats.columns = ['count', 'mean_days', 'median_days', 'surgery_rate', 'success_rate']
sport_stats = sport_stats.sort_values('mean_days', ascending=False)

print("Recovery Statistics by Sport:")
sport_stats

In [ ]:
# Visualize sport comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Mean recovery time
colors = plt.cm.RdYlGn_r(np.linspace(0.2, 0.9, len(sport_stats)))
sport_stats['mean_days'].plot(kind='bar', ax=axes[0], color=colors, edgecolor='white')
axes[0].set_ylabel('Mean Days Missed')
axes[0].set_title('Average Recovery Time by Sport')
axes[0].tick_params(axis='x', rotation=45)

# Success rate
sport_stats['success_rate'].plot(kind='bar', ax=axes[1], color='#2ecc71', edgecolor='white')
axes[1].set_ylabel('Recovery Success Rate')
axes[1].set_title('Recovery Success Rate by Sport')
axes[1].tick_params(axis='x', rotation=45)
axes[1].set_ylim(0.8, 1.0)

plt.tight_layout()
plt.savefig('../visualizations/sport_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Summary

In [ ]:
print("="*70)
print("SPORTS INJURY ANALYSIS - KEY FINDINGS")
print("="*70)

print("\n📊 DATASET OVERVIEW:")
print(f"   • {len(df)} injury records analyzed")
print(f"   • {df['sport'].nunique()} sports represented")
print(f"   • {df['injury_type'].nunique()} injury types")

print("\n🔬 HYPOTHESIS TESTING RESULTS:")

print(f"\n   H1: ACL vs Hamstring")
print(f"       • ACL mean: {acl_data.mean():.0f} days")
print(f"       • Hamstring mean: {hamstring_data.mean():.0f} days")
print(f"       • Difference: {acl_data.mean() - hamstring_data.mean():.0f} days")
print(f"       • Result: {'SUPPORTED' if p_value < 0.05 else 'NOT SUPPORTED'}")

print(f"\n   H2: Season Phase")
print(f"       • Playoffs mean: {playoffs.mean():.0f} days")
print(f"       • Regular season mean: {regular.mean():.0f} days")
print(f"       • Result: {'SUPPORTED' if p_val2 < 0.05 else 'NOT SUPPORTED'}")

print(f"\n   H3: Recurring Injuries")
print(f"       • Recurring mean: {recurring.mean():.0f} days")
print(f"       • First-time mean: {first_time.mean():.0f} days")
print(f"       • Result: {'SUPPORTED' if p_val3 < 0.05 else 'NOT SUPPORTED'}")

print("\n📈 KEY INSIGHTS:")
print(f"   • ACL injuries are the most severe (avg {injury_stats.loc['Knee ACL', 'mean_days']:.0f} days)")
print(f"   • Concussions are the quickest to recover (avg {injury_stats.loc['Concussion', 'mean_days']:.0f} days)")
print(f"   • Overall recovery success rate: {df['recovery_success'].mean():.1%}")

print("\n" + "="*70)